<a href="https://colab.research.google.com/github/onljunior72-dot/Proyecto/blob/main/ColabSteam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start ColabSteam below</b><br/>
<audio autoplay="" src="https://github.com/kmille36/Colab-Cloud-Gaming/raw/refs/heads/main/silence.m4a" loop controls>

In [ ]:
import os
import subprocess
import time
import select

print("="*60)
print("REPARACION DE TURN + CONTENIDO DE ESCRITORIO")
print("="*60)

# 1. Matar server actual
print("[1/4] Reiniciando servidor...")
os.system("pkill -9 -f 'python.*run.py'")
time.sleep(2)

# 2. Abrir ventana de contenido en el escritorio (para que haya algo que ver)
print("[2/4] Abriendo ventana de prueba...")
os.environ["DISPLAY"] = ":1"
# Abrir xterm con colores para verificar que hay contenido
subprocess.Popen(["xterm", "-bg", "black", "-fg", "green", "-fs", "14",
                 "-e", "echo 'SpaceLink Colab Activo' && bash"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 3. Parche CRÍTICO: Inyectar TURN directo en el código de WebRTC
print("[3/4] Aplicando parche TURN forzado...")

# Modificar webrtc_server.py para usar TURN obligatorio
webrtc_fix = '''
import json
import os

# TURN CONFIGURACION FORZADA
TURN_SERVERS = [
    {"urls": "stun:stun.l.google.com:19302"},
    {"urls": "turn:openrelay.metered.ca:80", "username": "openrelayproject", "credential": "openrelayproject"},
    {"urls": "turn:openrelay.metered.ca:443", "username": "openrelayproject", "credential": "openrelayproject"}
]

print("[WebRTC] TURN servers cargados:", len(TURN_SERVERS))
'''

# Leer archivo actual
with open("/content/SpaceLink/src/core/webrtc_server.py", "r") as f:
    content = f.read()

# Reemplazar todo el inicio hasta las importaciones
if "TURN_SERVERS" not in content:
    # Encontrar donde empiezan las importaciones reales (después de los comentarios/imports)
    lines = content.split('\n')
    new_lines = []
    insert_done = False

    for line in lines:
        if not insert_done and (line.startswith('import') or line.startswith('from')):
            new_lines.append(webrtc_fix)
            insert_done = True
        new_lines.append(line)

    with open("/content/SpaceLink/src/core/webrtc_server.py", "w") as f:
        f.write('\n'.join(new_lines))
    print("   Parche aplicado")

# 4. Reiniciar SpaceLink
print("[4/4] Reiniciando...")
os.chdir("/content/SpaceLink")
server = subprocess.Popen(
    ["python", "run.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    env=os.environ.copy()
)

time.sleep(5)

# Recrear tunnel
os.system("pkill cloudflared")
time.sleep(1)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

url = None
for _ in range(20):
    readable, _, _ = select.select([tunnel.stdout], [], [], 1)
    if readable:
        line = tunnel.stdout.readline()
        if "trycloudflare.com" in line:
            import re
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
            if m:
                url = m.group(0)
                break

print("\n" + "="*60)
if url:
    print(f"NUEVA URL: {url}/webrtc-test")
    print("\nINSTRUCCIONES:")
    print("1. Abre la URL en NAVEGADOR DE ESCRITORIO (Chrome/Edge)")
    print("2. NO uses el movil - el WebRTC movil es problematico")
    print("3. Espera 10 segundos y mira si aparece el xterm verde")
    print("4. Si sigue negro, prueba en modo incognito")
else:
    print("URL no capturada")
print("="*60)

# Logs
print("\nLogs:")
try:
    while True:
        readable, _, _ = select.select([server.stdout], [], [], 0.5)
        if readable:
            line = server.stdout.readline()
            if line and ("ICE" in line or "TURN" in line or "connected" in line.lower()):
                print(line.strip())
        time.sleep(0.5)
except:
    pass
